email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [16]:
from langchain_openrouter import ChatOpenRouter
# model = ChatOpenRouter(model="openai/gpt-4.1-nano")   
model = ChatOpenRouter(model="openai/gpt-5-nano")   

In [23]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    # "gpt-5-nano",
    model=model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [24]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

You're authenticated successfully and I’ve checked your inbox. Here’s what I see:

New message
- From: Jane (jane@example.com)
- Message: "Hi Julie, I'm going to be in town next week and was wondering if we could grab a coffee? - best, Jane"

What would you like to do next?
- Reply to this email (I can draft a reply for you)
- Mark as read
- Archive
- Delete
- Create a reminder to respond later

If you’d like a draft reply, tell me the tone you want (friendly, concise, detailed) and any times you’re available next week, and I’ll propose a couple of message options.


In [25]:
response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s a ready-to-send draft you can use:

Subject: Re: Coffee next week

Hi Jane,

That sounds wonderful — I’d love to grab a coffee. I’m available next week on Monday, Wednesday, or Friday afternoon. Let me know which day works for you and where you’d like to meet.

Best,
Julie

If you’d like any tweaks (tone, different availability, or a different closing), tell me and I’ll adjust.


In [26]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

In [27]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Here’s a ready-to-send draft you can use:

Subject: Re: Coffee next week

Hi Jane,

That sounds wonderful — I’d love to grab a coffee. I’m available next week on Monday, Wednesday, or Friday afternoon. Let me know which day works for you and where you’d like to meet.

Best,
Julie

If you’d like any tweaks (tone, different availability, or a different closing), tell me and I’ll adjust.


In [28]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='04090b40-55c8-4ac8-9296-bbb6e29c4288'),
              AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'gAAAAABqkBhALvSiMDHXiOFLE3z6e_GUikrTY8ons-rXKqfZ4hbAw0YBVHvh5fiVragrQ_IOgeOI7LXkwju1pxQasljZX-aBaWyVU49rfhf3GJrBDsdVQfphw8lnt1V8anXzb42oXDVpMkEvKBhwLtauBUdOhPEs85JSAPKCtpNuziPUrmWQrbJAzgTDGIBjKbUiHkw5EYs9sMJUVRtJlb2GDMWoOg9rMKWzR909sLv79qfCzrd1pEhXtIv8bwlFCj9FcmOFjrLFOZHpFzSQoVIVsu1IcBi8RW4saWsHilN1wcGGgX7mRtB9i6A_lK-Jrc7SI3INkfaOhNK5HO7_CmjrZoChTAcwu31hkDBfiLqoXUxoakG7bnAk4HKgoGbkZIgp_Pry6_2pWUIJVTKtnSulLOkv5gOIgI7bP37tcium4tGDTN_AnyYIph1gxOlJ-tnDajX_NY2t-8tQPXMHoJqUY0za1FDmt514UBYKotnCYaj1blNiWzNyz-OQXgrva1JUpll2lDapjurM7ms90PhdEMjP3BpD-ygMSCmK_R5sA52j2IL4FlmrnkXcgL4cp2itUwiOYaLmAk7qn6y9-biY8na_8I9-FXvaoAi4kvlZPpqcUojeIfj7aQOBQ0d_bKNx-_P_5t1QfrWMhAA5oLYEvQdSOlapu9qhMXw0zkr81wwdn52yfBX8JGH9Fhf0fNjh3DWtC0qJDoLEv0iHICmPSb62